## Importing Libraries

In [7]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re

## Setting up files

In [8]:
GENERATION_MODEL = "qwen3:1.7b"
FILES_EXTR = glob.glob("../Test_Files/Clinical_trials/clinical-trial_*.txt")
FILES_CONV = glob.glob("../Test_Files/Clinical_trials/Criteria_extracted/clinical-trial-extracted*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_trials/GT-clinical-trial_*.json")

PROMPT_EXTR_FILE = "./prompts/criteria-extraction_prompt.txt"
OUTPUT_EXTR_DIR = "./llm-outputs/criteria-extraction/"
OUTPUT_EXTR_FILE = "experiment"

PROMPT_CONV_FILE = "./prompts/criteria-conversion_prompt.txt"
OUTPUT_CONV_DIR = "./llm-outputs/criteria-conversion/"
OUTPUT_CONV_FILE = "experiment"

print(f"Found the following files for extraction - {FILES_EXTR}")
print(f"Found the following files for conversion - {FILES_CONV}")
print(f"Found the following golden diaries {GOLD_FILES}")

Found the following files for extraction - ['../Test_Files/Clinical_trials\\clinical-trial_e1.txt']
Found the following files for conversion - ['../Test_Files/Clinical_trials/Criteria_extracted\\clinical-trial-extracted_e1.txt']
Found the following golden diaries ['../Test_Files/Clinical_trials\\GT-clinical-trial_e1.json']


## Setting up environment

In [23]:
## Setting evironment
def set_env(prompt_file,output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
        base_prompt = " ".join(prompt_arr)

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, count

base_prompt_extr, count_extr_exp = set_env(PROMPT_EXTR_FILE, OUTPUT_EXTR_DIR)

base_prompt_conv, count_conv_exp = set_env(PROMPT_CONV_FILE, OUTPUT_CONV_DIR)
        


## Criteria Conversion
In this second phase the already extracted criteria in natural language of a given clinical trial will be converted into logical rules that way allowing the deterministic matching of patients with the clinical trial

In [25]:
pbar = tqdm(total=len(FILES_CONV), desc="Processing trials for criteria conversion")

for file in FILES_CONV:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt_conv.replace("{{TRIAL_TEXT}}",text)
        
        stream = chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            )
        
        llm_output = ""
        for chunk in stream:
            llm_output += chunk["message"]["content"]

        with open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{count_conv_exp}.txt","w",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_CONV_FILE}-{count_conv_exp}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria conversion:   0%|          | 0/1 [00:00<?, ?it/s]

processing file: ../Test_Files\clinical-trial-extracted_e1.txt


Processing trials for criteria conversion: 100%|██████████| 1/1 [05:41<00:00, 341.87s/it]

Saved LLM output on experiment-1




## Evaluation
In this phase the pipeline of extraction will be evaluated in 2 different fields:
- Correct classification (inclusion/exclusion)
- Logic correctness of rules

In [12]:
for gold_file in GOLD_FILES:
    with open(gold_file,"r",encoding="utf-8") as gf, \
         open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{1}.txt","r",encoding="utf-8") as out_extr, \
         open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{2}.txt","r",encoding="utf-8") as out_conv:
        
        curr_gf_trial = gold_file.split('_')[3].split('.')[0]
        
        print("Current trial: ", curr_gf_trial)

        data_gf = json.load(gf)

        out_extr_arr = [t.strip() for t in out_extr.readlines() if t.strip()]
        out_conv_arr = [t.strip() for t in out_conv.readlines() if t.strip()]

        out_extr_text = " ".join(out_extr_arr)
        out_conv_text = " ".join(out_conv_arr)

        outputs_extraction = out_extr_text.split("Ouput for file ")
        outputs_extraction.pop(0)

        outputs_conversion = out_conv_text.split("Ouput for file ")
        outputs_conversion.pop(0)
        
        print("outputs_extraction: ", outputs_extraction)
        print("outputs_conversion: ", outputs_conversion)
        
        for output_c, output_e in zip(outputs_conversion, outputs_extraction):
            if curr_gf_trial not in output_c or curr_gf_trial not in output_e:
                continue

            # Extract JSON safely
            json_conv_match = re.search(r"\{.*\}", output_c, flags=re.DOTALL)
            if not json_conv_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            json_extr_match = re.search(r"\{.*\}", output_e, flags=re.DOTALL)
            if not json_extr_match:
                print("No JSON found for", curr_gf_trial)
                continue
            
            output_conv_json = json.loads(json_conv_match.group(0))
            output_extr_json = json.loads(json_extr_match.group(0))

            print("Golden truth data ", data_gf)
            print("Output of the LLM (conversion) ", output_conv_json)
            print("Output of the LLM (extraction) ", output_extr_json)
            
            # Correct classification (inclusion/exclusion)
            
            gf_inclusion = set([c.strip().lower() for c in data_gf["inclusion_criteria"]])
            gf_exclusion = set([c.strip().lower() for c in data_gf["exclusion_criteria"]])

            llm_inclusion = set([c.strip().lower() for c in output_extr_json["inclusion_criteria"]])
            llm_exclusion = set([c.strip().lower() for c in output_extr_json["exclusion_criteria"]])
            
            correct = 0
            total = 0

            for crit in llm_inclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_inclusion):
                    correct += 1

            for crit in llm_exclusion:
                total += 1
                if any(crit in g or g in crit for g in gf_exclusion):
                    correct += 1

            accuracy = correct / total if total > 0 else 0

            print("Correct classification:", correct, "/", total)
            print("Accuracy:", round(accuracy, 4))

Current trial:  e1
outputs_extraction:  ['../Test_Files\\clinical-trial_e1.txt { "inclusion_criteria": [ "Age ≥ 18 years", "Histologically or cytologically confirmed metastatic NSCLC (Stage IV)", "Documented progression after first-line platinum-based chemotherapy combined with anti-PD-1 or anti-PD-L1 therapy", "ECOG Performance Status 0–1", "At least one measurable lesion per RECIST 1.1", "Adequate organ function: - Absolute neutrophil count (ANC) ≥ 1.5 x 10^9/L - Platelets ≥ 100 x 10^9/L - Hemoglobin ≥ 9 g/dL - AST and ALT ≤ 2.5 x ULN (≤ 5 x ULN if liver metastases) - Total bilirubin ≤ 1.5 x ULN - Creatinine clearance ≥ 40 mL/min (CKD-EPI formula)", "Women of childbearing potential must have a negative pregnancy test prior to treatment initiation", "Signed informed consent prior to any study-specific procedure" ], "exclusion_criteria": [ "Known EGFR, ALK, or ROS1 genomic alterations with available approved targeted therapy", "Untreated or symptomatic brain metastases", "Active autoim